In [2]:
import os
import csv
import yaml

In [3]:
def create_file_names_list(folder_path):
    file_list = os.listdir(folder_path)
    return file_list

def create_file_names_list_csv(csv_file_path, filter_func):
    with open(csv_file_path, "r") as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=',')
        file_list = [row[0] for row in csv_reader if filter_func(row)]
    return file_list

def compare_file_lists(files_currently_processed, files_to_be_processed):
    """Checking if all files in files_to_be_processed are in files_currently_processed, i.e., if all files that should be processed are processed. 
    If files_to_be_processed contains copies, they are ignored.
    
    Parameters:
    -----------
    files_currently_processed: list
        List of files that are currently processed.
    files_to_be_processed: list
        List of files that should be processed.

    Returns:
    --------
    missing_files: list
        List of files that are in files_to_be_processed but not in files_currently_processed.
    copied_files: list
        List of files that are copies.
    """
    missing_files = []
    copied_files = []
    for file in files_to_be_processed:
        if "copy" in file.lower():
            copied_files.append(file)
            continue
        if file not in files_currently_processed:
            missing_files.append(file)
    # sort the list
    missing_files.sort()
    copied_files.sort()
    return missing_files, copied_files

def get_missing_indices_gt(gt_files):
    """
    Get the missing indices in the ground truth files, i.e., indices that are not existing and break the incrementing sequence.
    """
    indices = file_name_to_index(gt_files)

    min_index = min(indices)
    max_index = max(indices)

    missing_indices = []
    for i in range(int(min_index), int(max_index)+1):
        if i not in indices:
            missing_indices.append(i)
    return missing_indices

def file_name_to_index(files):
    #extract the indices from the file names
    indices = []
    for file in files:
        index = file.split("_")[1]
        #remove the file extension
        index = index.split(".")[0]
        # remove the leading zeros
        index = index.lstrip("0")
        # ignore copies
        if "copy" in index.lower():
            continue
        indices.append(int(index))
    
    return sorted(indices)


def get_missing_indices_ranges(missing_files):
    # only extract lowest and highest index of following missing indices
    missing_indices = file_name_to_index(missing_files)
    start_index = missing_indices[0]
    end_index = None
    missing_ranges = []
    for i in range(len(missing_indices)-1):
        if missing_indices[i+1] - missing_indices[i] > 1:
            end_index = missing_indices[i]
            missing_ranges.append((start_index, end_index, end_index-start_index+1))
            start_index = missing_indices[i+1]

    # After the loop, close the last range
    end_index = missing_indices[-1]
    missing_ranges.append((start_index, end_index, end_index - start_index + 1))
    return missing_ranges
        
def get_completness_status(files_currently_processed, files_to_be_processed, blocks, last_range=None):
    missing_files, copied_files = compare_file_lists(files_currently_processed, files_to_be_processed)

    print("Copied files: ", copied_files)
    print("Missing files: ", missing_files)
    print("Number of missing files: ", len(missing_files))

    if missing_files:
        ranges = get_missing_indices_ranges(missing_files)
        
        print_block_status(missing_files, blocks)
        # print numbers that changed from former version in other color
        if last_range is not None:
            for r in ranges:
                if r in last_range:
                    print("\033[1;32m", r, "\033[0m")
                else:
                    print("\033[1;31m", r, "\033[0m")

        print("Missing ranges: ", ranges)


        last_range = ranges.copy()
    else:
        assert len(files_currently_processed) == len(files_to_be_processed) - len(copied_files), f'Number of files currently processed: {len(files_currently_processed)}, Number of files to be processed: {len(files_to_be_processed)}'
        print("All files processed.")

    return last_range
    
def divide_ranges(ranges, missing_files, num_blocks):
    # Step 1: Calculate total number of elements
    total_elements = sum([r[2] for r in ranges])
    
    # Step 2: Determine the ideal block size
    ideal_block_size = total_elements / num_blocks
    
    # Step 3: Combine ranges to form blocks
    blocks = []
    current_block_start = ranges[0][0]
    current_block_elements = 0
    mf_indices = file_name_to_index(missing_files)
    for i, (start, _, num_elements) in enumerate(ranges): 
        if current_block_elements + num_elements > ideal_block_size and len(blocks) < num_blocks - 1:
            # Close the current block and start a new one
            indices_in_block = [idx for idx in mf_indices if current_block_start <= idx <= ranges[i-1][1]]
            blocks.append((current_block_start, ranges[i-1][1], current_block_elements, indices_in_block))  # Use the previous range's end
            current_block_start = start
            current_block_elements = num_elements
        else:
            # Add to the current block
            current_block_elements += num_elements
    
    # Add the last block
    indices_in_block = [idx for idx in mf_indices if current_block_start <= idx <= ranges[-1][1]]
    blocks.append((current_block_start, ranges[-1][1], current_block_elements, indices_in_block))
    
    return blocks


def get_blocks(files_currently_processed, files_to_be_processed, num_blocks):
    missing_files, _ = compare_file_lists(files_currently_processed, files_to_be_processed)
    if not missing_files:
        print("All files processed.")
        return None
    ranges = get_missing_indices_ranges(missing_files)
    blocks = divide_ranges(ranges, missing_files, num_blocks)

    return blocks


def print_block_status(missing_files, blocks):
    """
    Returns the start, the current file, the end, and percentage of completness of the current block.
    """
    block_status = []
    mf_indices = file_name_to_index(missing_files)
    current_index =mf_indices[0]
    for block in blocks:
        if block[0] <= current_index <= block[1]:     
            elements_processed_in_block = block[3].index(current_index)
            block_status.append((block[0], current_index, block[1], block[2], elements_processed_in_block))
        # delete elements in mf_indices that are in the last block
        mf_indices = [i for i in mf_indices if i > block[1]]
        if mf_indices:
            current_index = mf_indices[0]
        else:
            break

    for i, status in enumerate(block_status):
        print(f"Block {i+1}: {status[0]} - \033[1;32m {status[1]} \033[0m - {status[2]}, {status[4]}/{status[3]} elements processed. ({status[4]/status[3]*100:.2f}%)")


    

# Load datasets

In [4]:
# Load the YAML file into a dictionary
dataset_yaml_file = '/workspace/home/data/datasets.yaml'
with open(dataset_yaml_file, 'r') as file:
    datasets = yaml.safe_load(file)

dataset_key ='ISIC2019'# 'Pascal_VOC07' #
dataset = datasets[dataset_key]

# Inpainted

In [15]:
id_num = "0001"
inpainted_folder_path = f"{dataset['inpainted_path']}/{id_num}"
seg_folder_path = f"{dataset['seg_path']}_dito/{id_num}"

if not os.path.exists(inpainted_folder_path):
    os.makedirs(inpainted_folder_path)

inpainted_files = create_file_names_list(inpainted_folder_path)
seg_files = create_file_names_list(seg_folder_path)

last_range = None
blocks = get_blocks(inpainted_files, seg_files, 10)

In [22]:
inpainted_files = create_file_names_list(inpainted_folder_path)
last_range = get_completness_status(inpainted_files, seg_files, blocks, last_range)


Copied files:  []
Missing files:  ['ISIC_0000067.jpg', 'ISIC_0000076.jpg', 'ISIC_0000080.jpg', 'ISIC_0000082.jpg', 'ISIC_0000083.jpg', 'ISIC_0000085.jpg', 'ISIC_0000090.jpg', 'ISIC_0000092.jpg', 'ISIC_0000098.jpg', 'ISIC_0000099.jpg', 'ISIC_0000100.jpg', 'ISIC_0000101.jpg', 'ISIC_0000104.jpg', 'ISIC_0000107.jpg', 'ISIC_0000108.jpg', 'ISIC_0000110.jpg', 'ISIC_0000112.jpg', 'ISIC_0000113.jpg', 'ISIC_0000119.jpg', 'ISIC_0000121.jpg', 'ISIC_0000130.jpg', 'ISIC_0000131.jpg', 'ISIC_0000133.jpg', 'ISIC_0000136.jpg', 'ISIC_0000143.jpg', 'ISIC_0000144.jpg', 'ISIC_0000145.jpg', 'ISIC_0000146.jpg', 'ISIC_0000149.jpg', 'ISIC_0000151.jpg', 'ISIC_0000156.jpg', 'ISIC_0000158.jpg', 'ISIC_0000170.jpg', 'ISIC_0000178.jpg', 'ISIC_0000179.jpg', 'ISIC_0000180.jpg', 'ISIC_0000184.jpg', 'ISIC_0000187.jpg', 'ISIC_0000219.jpg', 'ISIC_0000220.jpg', 'ISIC_0000224.jpg', 'ISIC_0000227.jpg', 'ISIC_0000231.jpg', 'ISIC_0000245.jpg', 'ISIC_0000247.jpg', 'ISIC_0000249.jpg', 'ISIC_0000250.jpg', 'ISIC_0000252.jpg', 'ISIC

# Segmentation

In [11]:
id_num = '0001'
dataset_key = 'ISIC2019'
seg_method = 'dito'

segmentation_path = f"/workspace/home/data/{dataset_key}/seg_{seg_method}/{id_num}"
sgl_pred_csv_path = f"/workspace/home/pipeline_outputs/{dataset_key}/{id_num}/sgl_pred.csv"

seg_files = create_file_names_list(segmentation_path)
sgl_pred_files = create_file_names_list_csv(sgl_pred_csv_path, lambda row: row[1] == "1")

last_range = None
blocks = get_blocks(seg_files, sgl_pred_files, 10)

All files processed.


In [12]:
seg_files = create_file_names_list(segmentation_path)

last_range = get_completness_status(seg_files, sgl_pred_files, blocks, last_range)

Copied files:  []
Missing files:  []
Number of missing files:  0
All files processed.
